<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width="400px" style="opacity:0.7">
</center>


In [1]:
%run supportvectors-common.ipynb



<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



# Contextual and Late Chunking

This demo demonstrates how semantic chunking can lose important context (like pronoun references) and how contextual chunking and late chunking can restore this lost context.

We'll use a sample text with pronoun references that will be lost during semantic chunking but restored through contextual or late chunking.


In [2]:
from rich import print as rprint

### Sample Text

Machine Learning Fundamentals

Neural networks are computational models inspired by biological neural networks. They consist of interconnected nodes called neurons that process information through weighted connections. The basic building block is the perceptron, which takes multiple inputs, applies weights, and produces an output through an activation function.

The learning process in neural networks involves adjusting these weights based on training data. This is typically done through backpropagation, where the network calculates the gradient of the loss function with respect to each weight and updates them accordingly. The learning rate determines how much the weights are adjusted in each iteration.

Deep learning extends this concept by using multiple hidden layers between the input and output layers. Each layer can learn increasingly complex features, with early layers detecting simple patterns like edges and later layers combining these into more complex concepts. This hierarchical feature learning is what makes deep neural networks so powerful for tasks like image recognition and natural language processing.

Advanced Optimization Techniques

While basic gradient descent works for simple problems, more sophisticated optimization algorithms have been developed to improve training efficiency and convergence. These techniques address common challenges like getting stuck in local minima, slow convergence, and handling different scales of gradients across parameters.

One popular approach is adaptive learning rates, where the learning rate is adjusted for each parameter based on its historical gradients. Adam (Adaptive Moment Estimation) combines the benefits of momentum and adaptive learning rates by maintaining exponentially decaying averages of both gradients and squared gradients. This allows the algorithm to automatically adjust the learning rate for each parameter, often leading to faster convergence and better performance.

Another important technique is regularization, which helps prevent overfitting by adding constraints to the model. L1 regularization adds a penalty proportional to the sum of absolute values of weights, encouraging sparsity. L2 regularization adds a penalty proportional to the sum of squared weights, encouraging smaller weights. Dropout randomly sets a fraction of input units to zero during training, forcing the network to not rely on any single neuron and improving generalization.

Batch normalization is another crucial technique that normalizes the inputs to each layer by adjusting and scaling the activations. This helps stabilize training by reducing internal covariate shift, allowing for higher learning rates and making the network less sensitive to initialization. It also acts as a regularizer, reducing the need for dropout in some cases.

In [3]:
# Sample text with pronoun references that will be lost during semantic chunking
sample_text = """
Machine Learning Fundamentals

Neural networks are computational models inspired by biological neural networks. They consist of interconnected nodes called neurons that process information through weighted connections. The basic building block is the perceptron, which takes multiple inputs, applies weights, and produces an output through an activation function.

The learning process in neural networks involves adjusting these weights based on training data. This is typically done through backpropagation, where the network calculates the gradient of the loss function with respect to each weight and updates them accordingly. The learning rate determines how much the weights are adjusted in each iteration.

Deep learning extends this concept by using multiple hidden layers between the input and output layers. Each layer can learn increasingly complex features, with early layers detecting simple patterns like edges and later layers combining these into more complex concepts. This hierarchical feature learning is what makes deep neural networks so powerful for tasks like image recognition and natural language processing.

Advanced Optimization Techniques

While basic gradient descent works for simple problems, more sophisticated optimization algorithms have been developed to improve training efficiency and convergence. These techniques address common challenges like getting stuck in local minima, slow convergence, and handling different scales of gradients across parameters.

One popular approach is adaptive learning rates, where the learning rate is adjusted for each parameter based on its historical gradients. Adam (Adaptive Moment Estimation) combines the benefits of momentum and adaptive learning rates by maintaining exponentially decaying averages of both gradients and squared gradients. This allows the algorithm to automatically adjust the learning rate for each parameter, often leading to faster convergence and better performance.

Another important technique is regularization, which helps prevent overfitting by adding constraints to the model. L1 regularization adds a penalty proportional to the sum of absolute values of weights, encouraging sparsity. L2 regularization adds a penalty proportional to the sum of squared weights, encouraging smaller weights. Dropout randomly sets a fraction of input units to zero during training, forcing the network to not rely on any single neuron and improving generalization.

Batch normalization is another crucial technique that normalizes the inputs to each layer by adjusting and scaling the activations. This helps stabilize training by reducing internal covariate shift, allowing for higher learning rates and making the network less sensitive to initialization. It also acts as a regularizer, reducing the need for dropout in some cases.
"""

rprint("Sample text loaded with pronoun references:")
rprint(sample_text[:200] + "...")

Sample text loaded with pronoun references:

Machine Learning Fundamentals

Neural networks are computational models inspired by biological neural networks. They consist of interconnected 
nodes called neurons that process information through we...

## Step 1: Use Entire Document as Single Parent Chunk

We'll use the entire document as one parent chunk and then apply semantic chunking to it.


In [4]:
# Use entire document as single parent chunk
parent_chunk = {
    "id": 0,
    "text": sample_text.strip(),
    "title": "Complete Document"
}

rprint("Using entire document as single parent chunk:")
rprint(f"Length: {len(parent_chunk['text'])} characters")
rprint(f"Preview: {parent_chunk['text'][:200]}...")

Using entire document as single parent chunk:

Length: 2823 characters

Preview: Machine Learning Fundamentals

Neural networks are computational models inspired by biological neural networks. They consist of interconnected 
nodes called neurons that process information through wei...

## Step 2: Semantic Chunking

Use `chonkie` to semantically chunk each parent chunk. This will break down the text into smaller, semantically coherent pieces, but may lose important context like pronoun references.


In [5]:
from chonkie import SemanticChunker

# Initialize semantic chunker
semantic_chunker = SemanticChunker()

# Process the single parent chunk
semantic_chunks = []
parent_chunk_dict = {
    "id": parent_chunk["id"],
    "text": parent_chunk["text"],
    "title": parent_chunk["title"],
    "semantic_chunks": []
}

# Get semantic chunks for the parent
sem_chunks = semantic_chunker.chunk(parent_chunk["text"])

for i, sc in enumerate(sem_chunks):
    semantic_chunk_dict_item = {
        "id": i,
        "text": sc.text,
        "start_char": sc.start_index,
        "end_char": sc.end_index
    }
    parent_chunk_dict["semantic_chunks"].append(semantic_chunk_dict_item)
    
    # Also maintain a flat list for contextual chunking
    semantic_chunks.append({
        "chunk_id": i,
        "chunk": sc,
        "parent_id": parent_chunk["id"],
        "parent_chunk": parent_chunk
    })


In [6]:

import pandas as pd
from IPython.display import display

# Pandas options: don't truncate long strings; use full available width
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

# Create a DataFrame from the semantic chunks
semantic_chunks_df = pd.DataFrame(
    [
        {
            "chunk_id": sc["chunk_id"],
            "chunk_text": sc["chunk"].text,
            "start_index": sc["chunk"].start_index,
            "end_index": sc["chunk"].end_index,
        }
        for sc in semantic_chunks
    ]
)

# Display: wrap chunk_text and stretch table width
display(
    semantic_chunks_df.head()
    .style.hide(axis="index")
    .set_properties(subset=["chunk_text"], **{"white-space": "pre-wrap"})
    .set_table_styles(
        [
            {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed")]},
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "td", "props": [("vertical-align", "top")]},
        ]
    )
)


chunk_id,chunk_text,start_index,end_index
0,"Machine Learning Fundamentals Neural networks are computational models inspired by biological neural networks. They consist of interconnected nodes called neurons that process information through weighted connections. The basic building block is the perceptron, which takes multiple inputs, applies weights, and produces an output through an activation function. The learning process in neural networks involves adjusting these weights based on training data. This is typically done through backpropagation, where the network calculates the gradient of the loss function with respect to each weight and updates them accordingly. The learning rate determines how much the weights are adjusted in each iteration.",0,714
1,Deep learning extends this concept by using multiple hidden layers between the input and output layers.,714,818
2,"Each layer can learn increasingly complex features, with early layers detecting simple patterns like edges and later layers combining these into more complex concepts. This hierarchical feature learning is what makes deep neural networks so powerful for tasks like image recognition and natural language processing.",818,1135
3,Advanced Optimization Techniques,1135,1169
4,"While basic gradient descent works for simple problems, more sophisticated optimization algorithms have been developed to improve training efficiency and convergence.",1169,1336


## Step 3: Identify Pronoun References Lost in Semantic Chunking

Let's examine some semantic chunks to see where pronoun references like "it", "this", "they" lose their context.


In [7]:
# Find semantic chunks with pronoun references
pronoun_chunks = []

for i, sc in enumerate(semantic_chunks):
    text = sc['chunk'].text.lower()
    pronouns = ['it ', 'this ', 'they ', 'these ', 'that ', 'those ']
    found_pronouns = [p for p in pronouns if p in text]
    
    if found_pronouns:
        pronoun_chunks.append({
            'chunk_id': i,
            'text': sc['chunk'].text,
            'pronouns': found_pronouns
        })


In [8]:

import pandas as pd
from IPython.display import display

# Pandas options: don't truncate long strings; use full available width
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

pronoun_chunks_df = pd.DataFrame(
    [
        {
            "chunk_id": c["chunk_id"],
            "pronouns": c["pronouns"],
            "chunk_text": c["text"],
        }
        for c in pronoun_chunks
    ]
)

display(
    pronoun_chunks_df.head(3)
    .style.hide(axis="index")
    .set_properties(subset=["chunk_text"], **{"white-space": "pre-wrap"})
    .set_table_styles(
        [
            {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed")]},
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "td", "props": [("vertical-align", "top")]},
        ]
    )
)


chunk_id,pronouns,chunk_text
0,"['this ', 'they ', 'these ', 'that ']","Machine Learning Fundamentals Neural networks are computational models inspired by biological neural networks. They consist of interconnected nodes called neurons that process information through weighted connections. The basic building block is the perceptron, which takes multiple inputs, applies weights, and produces an output through an activation function. The learning process in neural networks involves adjusting these weights based on training data. This is typically done through backpropagation, where the network calculates the gradient of the loss function with respect to each weight and updates them accordingly. The learning rate determines how much the weights are adjusted in each iteration."
1,['this '],Deep learning extends this concept by using multiple hidden layers between the input and output layers.
2,"['this ', 'these ']","Each layer can learn increasingly complex features, with early layers detecting simple patterns like edges and later layers combining these into more complex concepts. This hierarchical feature learning is what makes deep neural networks so powerful for tasks like image recognition and natural language processing."


## Step 4: Contextual Chunking

Now we'll use an LLM to enrich each semantic chunk with context from its parent chunk, resolving pronoun references and making each chunk self-standing.


In [9]:
from openai import OpenAI

# Initialize OpenAI client 
client = OpenAI(
    base_url="http://10.0.10.51:8000/v1",
    api_key="dummy"
)

# Contextual chunking prompt
contextual_prompt = """
Here is the parent section: {parent_text}

Now here is the semantic chunk: {semantic_text}

Please produce an enriched chunk which retains the semantic chunk but adds any necessary context from the parent so that the chunk is self-standing. Pay special attention to resolving any pronoun references (it, this, they, etc.) by replacing them with their proper antecedents.

Do not add anything that is not needed to make the chunk self-standing.
"""

# Process each semantic chunk
contextual_chunks = []

rprint("Processing semantic chunks with contextual chunking...")
rprint("="*60)

for i, sc in enumerate(semantic_chunks):
    parent_text = sc['parent_chunk']['text']
    semantic_text = sc['chunk'].text
    
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "user", "content": contextual_prompt.format(parent_text=parent_text, semantic_text=semantic_text)},
            ],
        )
        
        contextual_chunk = response.choices[0].message.content
        
        contextual_chunks.append({
            "chunk_id": sc['chunk_id'],
            "parent_id": sc['parent_id'],
            "original_semantic_chunk": semantic_text,
            "contextual_chunk": contextual_chunk,
            "parent_title": sc['parent_chunk']['title']
        })
            
    except Exception as e:
        print(f"Error processing chunk {i}: {e}")
        contextual_chunks.append({
            "chunk_id": sc['chunk_id'],
            "parent_id": sc['parent_id'],
            "original_semantic_chunk": semantic_text,
            "contextual_chunk": semantic_text,  # fallback to original
            "parent_title": sc['parent_chunk']['title']
        })

rprint(f"\nCompleted contextual chunking for {len(contextual_chunks)} chunks")


Processing semantic chunks with contextual chunking...

============================================================

Completed contextual chunking for 10 chunks

## Step 5: Compare Original vs Contextual Chunks

Let's compare some of the original semantic chunks with their contextual versions to see how pronoun references were resolved.


In [10]:
# Compare original vs contextual chunks, focusing on those with pronoun references
rprint("COMPARISON: Original Semantic Chunks vs Contextual Chunks")
rprint("="*80)


COMPARISON: Original Semantic Chunks vs Contextual Chunks

================================================================================

In [11]:

import pandas as pd
from IPython.display import display

# Pandas options: don't truncate long strings; use full available width
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

# Build a DataFrame for chunks that originally had pronoun references
rows = []
for pronoun_chunk in pronoun_chunks:
    chunk_id = pronoun_chunk["chunk_id"]
    contextual_chunk = next((cc for cc in contextual_chunks if cc["chunk_id"] == chunk_id), None)
    if contextual_chunk:
        rows.append(
            {
                "chunk_id": chunk_id,
                "pronouns_found": ", ".join(pronoun_chunk["pronouns"]),
                "original_chunk": contextual_chunk["original_semantic_chunk"],
                "contextual_chunk": contextual_chunk["contextual_chunk"],
            }
        )

comparison_df = pd.DataFrame(rows, columns=[
    "chunk_id",
    "pronouns_found",
    "original_chunk",
    "contextual_chunk",
])

# Display as full-width table with wrapped text for the chunk columns
display(
    comparison_df
    .style.hide(axis="index")
    .set_properties(
        subset=["original_chunk", "contextual_chunk"],
        **{"white-space": "pre-wrap", "word-wrap": "break-word"},
    )
    .set_table_styles(
        [
            {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed")]},
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "td", "props": [("vertical-align", "top")]},
        ]
    )
)


chunk_id,pronouns_found,original_chunk,contextual_chunk
0,"this , they , these , that","Machine Learning Fundamentals Neural networks are computational models inspired by biological neural networks. They consist of interconnected nodes called neurons that process information through weighted connections. The basic building block is the perceptron, which takes multiple inputs, applies weights, and produces an output through an activation function. The learning process in neural networks involves adjusting these weights based on training data. This is typically done through backpropagation, where the network calculates the gradient of the loss function with respect to each weight and updates them accordingly. The learning rate determines how much the weights are adjusted in each iteration.","**Enriched Chunk: Machine Learning Fundamentals** Neural networks are computational models inspired by biological neural networks. They consist of interconnected nodes called neurons that process information through weighted connections. The basic building block is the perceptron, a simple neuron that takes multiple inputs, applies a weight to each input, sums the weighted inputs, and then produces an output through an activation function. The learning process in neural networks involves adjusting these weights based on training data. This adjustment is typically performed using backpropagation, a method that computes the gradient of a loss function with respect to every weight in the network. The loss function is a differentiable measure of the discrepancy between the network’s predictions and the true target values; its gradient tells us how the loss changes in response to infinitesimal changes in each weight. Weight updates are then applied by subtracting the product of a learning rate and the corresponding gradient from each weight. The learning rate is a scalar that determines how large the adjustment to a weight is in each iteration, thereby controlling the step size of the update and influencing convergence speed and stability."
1,this,Deep learning extends this concept by using multiple hidden layers between the input and output layers.,"Neural networks are computational models inspired by biological neural networks, consisting of interconnected neurons that process information through weighted connections. Deep learning extends the concept of neural networks by using multiple hidden layers between the input and output layers."
2,"this , these","Each layer can learn increasingly complex features, with early layers detecting simple patterns like edges and later layers combining these into more complex concepts. This hierarchical feature learning is what makes deep neural networks so powerful for tasks like image recognition and natural language processing.","Deep neural networks consist of an input layer, multiple hidden layers, and an output layer. In these networks, each hidden layer can learn increasingly complex features from its input. The first hidden layers often detect simple patterns such as edges in images; deeper layers combine those patterns into more complex concepts like object shapes or sentence structures. This hierarchical feature learning is what gives deep neural networks their power for tasks such as image recognition and natural language processing."
5,these,"These techniques address common challenges like getting stuck in local minima, slow convergence, and handling different scales of gradients across parameters.","Advanced optimization techniques such as adaptive learning rates, Adam, regularization methods (L1, L2, and dropout), and batch normalization address common challenges during neural‑network training, including getting stuck in local minima, slow convergence, and handling different scales of gradients across parameters."
6,this,"One popular approach is adaptive learning rates, where the learning rate is adjusted for each parameter based on its historical gradients. Adam (Adaptive Moment Estimation) com

## Step 6: Late Chunking

Now let's implement late chunking as an alternative approach. Late chunking uses embeddings to capture context from the parent chunk and associate it with each semantic chunk.


In [12]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load embedding model
model_name = "jinaai/jina-embeddings-v2-base-en"
rprint(f"Loading embedding model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, output_hidden_states=True)

def late_chunk_parent(parent_chunk):
    """
    parent_chunk: dict with keys {id, text, semantic_chunks: List[{start_char, end_char, text, id}]}
    Returns enriched semantic chunks with embeddings from parent context.
    """
    text = parent_chunk["text"]
    sem_chunks = parent_chunk["semantic_chunks"]

    # Tokenize + embed the *parent chunk text only*
    inputs = tokenizer(text, return_tensors="pt", truncation=False, return_offsets_mapping=True)
    # Save offsets separately
    offsets = inputs.pop("offset_mapping")[0].tolist()
    
    with torch.no_grad():
        outputs = model(**inputs)
    token_embs = outputs.last_hidden_state.squeeze(0)

    enriched_semantics = []
    for sc in sem_chunks:
        s, e = sc["start_char"], sc["end_char"]
        indices = [i for i, (ts, te) in enumerate(offsets) if te > s and ts < e]
        if not indices:
            # If no tokens found, use mean of all parent embeddings
            emb = token_embs.mean(dim=0).cpu().numpy()
        else:
            emb = token_embs[indices].mean(dim=0).cpu().numpy()
            
        enriched_semantics.append({
            "semantic_id": sc["id"],
            "parent_id": parent_chunk["id"],
            "embedding": emb.tolist(),
            "text": sc["text"],
            "parent_text": text,
            "num_tokens": len(indices) if indices else 0
        })
    return enriched_semantics

rprint("Late chunking implementation ready!")


Loading embedding model: jinaai/jina-embeddings-v2-base-en

Late chunking implementation ready!

### Alternative: API-based late chunking (SV cluster)

Instead of loading the Jina model locally, we can call the SupportVectors cluster embedding API with `return_token_embeddings=True`. The API returns per-token embedding vectors — a list of lists of floats, where the outer list is one entry per token and each inner list is that token's embedding — rather than a single pooled vector for the entire input.

We still use the tokenizer locally only for character-to-token offset mapping, then average the overlapping token embeddings for each semantic chunk. The late-chunking logic is otherwise the same as the local approach above.

In [13]:
import os

import requests
from transformers import AutoTokenizer

# Same Jina model; embeddings are computed on the SV cluster instead of locally.
model_name = "jinaai/jina-embeddings-v2-base-en"
embed_host = os.getenv("EMBEDDING_HOST", "10.0.10.51")
embed_port = os.getenv("EMBEDDING_PORT", "8000")
embed_text_base_url = f"http://{embed_host}:{embed_port}/embed-text/v1/embeddings"

# Tokenizer is still needed locally for character-to-token offset mapping.
rprint(f"Loading tokenizer for offset mapping: {model_name}")
tokenizer_api = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)


def late_chunk_parent_api(parent_chunk):
    """
    parent_chunk: dict with keys {id, text, semantic_chunks: List[{start_char, end_char, text, id}]}
    Returns enriched semantic chunks with embeddings from parent context via the SV cluster API.
    """
    text = parent_chunk["text"]
    sem_chunks = parent_chunk["semantic_chunks"]

    inputs = tokenizer_api(
        text,
        return_tensors="pt",
        truncation=False,
        return_offsets_mapping=True,
    )
    offsets = inputs["offset_mapping"][0].cpu().tolist()

    payload = {
        "model": model_name,
        "input": [text],
        "return_token_embeddings": True,
    }
    response = requests.post(embed_text_base_url, json=payload)
    response.raise_for_status()

    # Per-token embeddings: list[list[float]] — one vector per input token.
    raw = response.json()["data"][0]["embedding"]
    token_embs = raw[0] if (raw and isinstance(raw[0][0], list)) else raw

    enriched_semantics = []
    for sc in sem_chunks:
        s, e = sc["start_char"], sc["end_char"]
        indices = [i for i, (ts, te) in enumerate(offsets) if te > s and ts < e]
        if not indices:
            emb = [sum(x) / len(token_embs) for x in zip(*token_embs)]
        else:
            emb_vectors = [token_embs[i] for i in indices]
            emb = [sum(x) / len(x) for x in zip(*emb_vectors)]

        enriched_semantics.append({
            "semantic_id": sc["id"],
            "parent_id": parent_chunk["id"],
            "embedding": emb,
            "text": sc["text"],
            "parent_text": text,
            "num_tokens": len(indices) if indices else len(token_embs),
        })
    return enriched_semantics


rprint("API-based late chunking implementation ready!")

Loading tokenizer for offset mapping: jinaai/jina-embeddings-v2-base-en

API-based late chunking implementation ready!

In [14]:
# Process the single parent chunk with API-based late chunking
rprint("Processing parent chunk with API-based late chunking...")
rprint("=" * 50)

rprint(f"Processing parent chunk: {parent_chunk_dict['title']}")
enriched_semantics_api = late_chunk_parent_api(parent_chunk_dict)
all_late_chunks_api = enriched_semantics_api

Processing parent chunk with API-based late chunking...

==================================================

Processing parent chunk: Complete Document

In [15]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

late_chunks_api_df = pd.DataFrame(
    [
        {
            "chunk_id": es["semantic_id"],
            "chunk_text": es["text"],
            "embedding": es["embedding"][:10] + ["..."],
            "num_tokens": es["num_tokens"],
        }
        for es in enriched_semantics_api
    ]
)

display(
    late_chunks_api_df.head(3)
    .style.hide(axis="index")
    .set_properties(subset=["chunk_text"], **{"white-space": "pre-wrap"})
    .set_table_styles(
        [
            {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed")]},
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "td", "props": [("vertical-align", "top")]},
        ]
    )
)

chunk_id,chunk_text,embedding,num_tokens
0,"Machine Learning Fundamentals Neural networks are computational models inspired by biological neural networks. They consist of interconnected nodes called neurons that process information through weighted connections. The basic building block is the perceptron, which takes multiple inputs, applies weights, and produces an output through an activation function. The learning process in neural networks involves adjusting these weights based on training data. This is typically done through backpropagation, where the network calculates the gradient of the loss function with respect to each weight and updates them accordingly. The learning rate determines how much the weights are adjusted in each iteration.","[-0.4015779825858772, -0.8685385346239022, 0.3322494575582145, -0.26359129692348887, 0.32671814001662564, 0.15736817676964673, 0.0907669622871346, 0.031216436918130367, -0.06870895442533133, 0.5214554633100722, '...']",118
1,Deep learning extends this concept by using multiple hidden layers between the input and output layers.,"[-0.6035558298394522, -1.0251779398497414, 0.33275871066486135, -0.07376255763365942, 0.3040709281888078, 0.31122551321545067, 0.12478026145083063, -0.188571263652514, -0.21146236688775175, 0.3654667793581372, '...']",17
2,"Each layer can learn increasingly complex features, with early layers detecting simple patterns like edges and later layers combining these into more complex concepts. This hierarchical feature learning is what makes deep neural networks so powerful for tasks like image recognition and natural language processing.","[-0.723289281091032, -1.0919604059308767, 0.4035668581103285, -0.12275444625993259, 0.20239652530290186, 0.18852416192021337, -0.20311166967924996, -0.4310707778398258, -0.3492256544219951, 0.5352119982417207, '...']",48


In [16]:
# Process the single parent chunk with late chunking
rprint("Processing parent chunk with late chunking...")
rprint("="*50)

rprint(f"Processing parent chunk: {parent_chunk_dict['title']}")
enriched_semantics = late_chunk_parent(parent_chunk_dict)
all_late_chunks = enriched_semantics


Processing parent chunk with late chunking...

==================================================

Processing parent chunk: Complete Document

In [17]:

import pandas as pd
from IPython.display import display

# Configure pandas display options for better readability
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)

# Build a DataFrame summarizing late chunks
late_chunks_df = pd.DataFrame(
    [
        {
            "chunk_id": es["semantic_id"],
            "chunk_text": es["text"],
            # Show only the first 10 dimensions of the 768-dim embedding
            "embedding": es["embedding"][:10] + ["..."],
            "num_tokens": es["num_tokens"],
        }
        for es in enriched_semantics
    ]
)

# Display first few late chunks with nicely wrapped text
display(
    late_chunks_df.head(3)
    .style.hide(axis="index")
    .set_properties(subset=["chunk_text"], **{"white-space": "pre-wrap"})
    .set_table_styles(
        [
            {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed")]},
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "td", "props": [("vertical-align", "top")]},
        ]
    )
)


chunk_id,chunk_text,embedding,num_tokens
0,"Machine Learning Fundamentals Neural networks are computational models inspired by biological neural networks. They consist of interconnected nodes called neurons that process information through weighted connections. The basic building block is the perceptron, which takes multiple inputs, applies weights, and produces an output through an activation function. The learning process in neural networks involves adjusting these weights based on training data. This is typically done through backpropagation, where the network calculates the gradient of the loss function with respect to each weight and updates them accordingly. The learning rate determines how much the weights are adjusted in each iteration.","[-0.4015781581401825, -0.8685381412506104, 0.3322494626045227, -0.26359084248542786, 0.3267180919647217, 0.15736795961856842, 0.09076698869466782, 0.031216584146022797, -0.06870933622121811, 0.5214550495147705, '...']",118
1,Deep learning extends this concept by using multiple hidden layers between the input and output layers.,"[-0.6035559773445129, -1.0251779556274414, 0.33275875449180603, -0.07376150786876678, 0.30407023429870605, 0.3112257719039917, 0.12478054314851761, -0.1885712891817093, -0.2114620804786682, 0.36546677350997925, '...']",17
2,"Each layer can learn increasingly complex features, with early layers detecting simple patterns like edges and later layers combining these into more complex concepts. This hierarchical feature learning is what makes deep neural networks so powerful for tasks like image recognition and natural language processing.","[-0.7232891917228699, -1.091960072517395, 0.40356704592704773, -0.12275420874357224, 0.2023962140083313, 0.18852441012859344, -0.20311154425144196, -0.43107128143310547, -0.3492259681224823, 0.5352118611335754, '...']",48
